In [1]:
import torch

In [2]:
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)


In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/emotions-dataset-for-nlp/val.txt
/kaggle/input/emotions-dataset-for-nlp/test.txt
/kaggle/input/emotions-dataset-for-nlp/train.txt


In [4]:
#!pip uninstall -y pyarrow datasets transformers
#!pip install -U transformers datasets pyarrow

In [5]:
!pip install -U transformers accelerate peft safetensors


In [6]:
train_df = pd.read_csv(
    "/kaggle/input/emotions-dataset-for-nlp/train.txt",
    sep=";",
    names=["text", "label"]
)

val_df = pd.read_csv(
    "/kaggle/input/emotions-dataset-for-nlp/val.txt",
    sep=";",
    names=["text", "label"]
)

test_df = pd.read_csv(
    "/kaggle/input/emotions-dataset-for-nlp/test.txt",
    sep=";",
    names=["text", "label"]
)

train_df.head()


,text,label
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [7]:
val_df.head()

,text,label
0,im feeling quite sad and sorry for myself but ...,sadness
1,i feel like i am still looking at a blank canv...,sadness
2,i feel like a faithful servant,love
3,i am just feeling cranky and blue,anger
4,i can have for a treat or if i am feeling festive,joy


In [8]:
test_df.head()

,text,label
0,im feeling rather rotten so im not very ambiti...,sadness
1,im updating my blog because i feel shitty,sadness
2,i never make her separate from me because i do...,sadness
3,i left with my bouquet of red and yellow tulip...,joy
4,i was feeling a little vain when i did this one,sadness


In [9]:
#xtrain=train_df.text

In [10]:
#xtrain

In [11]:
label_list = sorted(train_df["label"].unique())

label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

In [12]:
train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"]   = val_df["label"].map(label2id)
test_df["label_id"]  = test_df["label"].map(label2id)

In [35]:
train_df = train_df[["text", "label_id"]].rename(columns={"label_id": "label"})
val_df   = val_df[["text", "label_id"]].rename(columns={"label_id": "label"})
test_df  = test_df[["text", "label_id"]].rename(columns={"label_id": "label"})

In [36]:
train_df

,text,label
0,i didnt feel humiliated,4
1,i can go from feeling so hopeless to so damned...,4
2,im grabbing a minute to post i feel greedy wrong,0
3,i am ever feeling nostalgic about the fireplac...,3
4,i am feeling grouchy,0
...,...,...
15995,i just had a very brief time in the beanbag an...,4
15996,i am now turning and i feel pathetic that i am...,4
15997,i feel strong and good overall,2
15998,i feel like this was such a rude comment and i...,0


### Converting pandas DataFrame to Hugging Face Dataset

The pandas DataFrame is converted into a Hugging Face `Dataset` object.
This step does **not** change the data itself, but enables:
- efficient batched preprocessing
- seamless integration with the Hugging Face Trainer
- optimized data loading for GPU training


In [38]:
from datasets import Dataset

#train_ds = pd.DataFrame({
    #"text": xtrain,
    #"label": ytrain
#})

train_ds = Dataset.from_pandas(train_df)


In [39]:
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)


In [30]:
#from transformers import BertTokenizerFast

#tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
#lets use Autotokenizer because it fits for amy model not only bert

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Now, we are gonna tokenize our texts.From words to token IDS.For that purpose, we use Autotokenizer/BertTokenizer pretrained from huggingface

In [17]:
def tokenize(batch):#since batch must not be a python dataframe ,we have converted it into a hugging face dataset
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [40]:
train_ds= train_ds.map(tokenize, batched=True)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

In [41]:
test_ds= test_ds.map(tokenize, batched=True)
val_ds= val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

### BERT Model for Sequence Classification

A pre-trained BERT model is loaded with an added classification head.
The number of output labels is set to match the number of emotion classes.

This allows the model to be fine-tuned specifically for emotion detection.


In [42]:
train_ds

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 16000
})

In [20]:
from transformers import AutoModelForSequenceClassification

Loads pretrained BERT weights

Adds a classification head on top

Output size = number of emotions (6 in your case)

📌 The label2id / id2label part is for readable predictions, not training logic.

In [21]:
model=AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

In [43]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [25]:
from transformers import Trainer

In [44]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [45]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.220178,0.206082,0.928500,0.929206
2,0.106917,0.169611,0.936000,0.935301
3,0.089205,0.166157,0.939500,0.939601


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3000, training_loss=0.23271261390050252, metrics={'train_runtime': 596.9277, 'train_samples_per_second': 80.412, 'train_steps_per_second': 5.026, 'total_flos': 3157446057984000.0, 'train_loss': 0.23271261390050252, 'epoch': 3.0})

In [46]:
trainer.evaluate(test_ds)


{'eval_loss': 0.18809229135513306,
 'eval_accuracy': 0.927,
 'eval_f1': 0.927355562302764,
 'eval_runtime': 7.3502,
 'eval_samples_per_second': 272.102,
 'eval_steps_per_second': 17.006,
 'epoch': 3.0}

In [47]:
trainer.save_model("emotion_bert_model")
tokenizer.save_pretrained("emotion_bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('emotion_bert_model/tokenizer_config.json',
 'emotion_bert_model/tokenizer.json')